# Deep Autoencoder — Reducing the Dimensionality of Data with Neural Networks (2006)

_Papers · Self-supervised & Representation_

**A deep network with a tiny middle layer learns a low-dimensional code that reconstructs the input — beating PCA at dimensionality reduction.**

---

This notebook is a practice scaffold for the **Deep Autoencoder — Reducing the Dimensionality of Data with Neural Networks (2006)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np
import torchvision, torchvision.transforms as T
torch.manual_seed(0); np.random.seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

# --- 0. Sanity-check the worked example: reconstruction MSE on a 4-pixel toy image. ---
x_toy  = torch.tensor([1.0, 0.0, 0.8, 0.2])
xh_toy = torch.tensor([0.9, 0.1, 0.6, 0.2])
sse = ((x_toy - xh_toy)**2).sum()
mse = ((x_toy - xh_toy)**2).mean()
print("worked example:  sum of squared errors =", round(sse.item(), 4),
      " MSE =", round(mse.item(), 4))
# worked example:  sum of squared errors = 0.06   MSE = 0.015


# --- 1. The deep autoencoder: encoder -> 2-D code (bottleneck) -> decoder. ---
class AutoEncoder(nn.Module):
    def __init__(self, code=2, linear=False):     # linear=True -> ablation (= PCA)
        super().__init__()
        act = nn.Identity if linear else nn.ReLU  # remove nonlinearity for the ablation
        self.enc = nn.Sequential(nn.Linear(784,256), act(), nn.Linear(256,64), act(),
                                 nn.Linear(64, code))
        self.dec = nn.Sequential(nn.Linear(code,64), act(), nn.Linear(64,256), act(),
                                 nn.Linear(256,784), nn.Sigmoid())
    def encode(self, x): return self.enc(x)       # the low-dimensional code z
    def forward(self, x):
        z = self.enc(x); return self.dec(z)       # reconstruction x-hat


# --- 2. An MNIST subset (torchvision, preinstalled). Flatten to 784, values in [0,1]. ---
tf  = T.Compose([T.ToTensor(), T.Lambda(lambda t: t.view(-1))])
raw = torchvision.datasets.MNIST("./data", train=True, download=True, transform=tf)
idx = np.random.RandomState(0).permutation(len(raw))[:6000]
X   = torch.stack([raw[i][0] for i in idx]).to(device)         # (6000, 784)
y   = torch.tensor([raw[i][1] for i in idx])                   # labels: ONLY for plotting/eval


def train_ae(linear=False, epochs=40, lr=1e-3):
    torch.manual_seed(0)
    ae = AutoEncoder(code=2, linear=linear).to(device)
    opt = torch.optim.Adam(ae.parameters(), lr=lr); loss_fn = nn.MSELoss()
    ae.train(); B = 256
    for ep in range(epochs):
        perm = torch.randperm(len(X)); tot = 0.0; nb = 0
        for s in range(0, len(X), B):
            xb = X[perm[s:s+B]]
            xh = ae(xb); loss = loss_fn(xh, xb)                # reconstruct the INPUT (no label)
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item(); nb += 1
        if ep % 10 == 0: print(f"  epoch {ep:2d}  recon MSE {tot/nb:.4f}")
    return ae

print("\n=== train deep (nonlinear) autoencoder ===")
ae = train_ae(linear=False)
with torch.no_grad():
    Z_ae = ae.encode(X).cpu().numpy()                          # 2-D autoencoder codes


# --- 3. PCA baseline: first two principal components of the SAME flattened images. ---
Xc = (X - X.mean(0)).cpu()
U, S, V = torch.pca_lowrank(Xc, q=2)
Z_pca = (Xc @ V).numpy()                                       # 2-D PCA embedding


# --- 4. Ablation: how well does a simple classifier do on each 2-D code? (separation metric) ---
def knn_acc(Z, y, k=15):                                       # 1-fold k-NN accuracy on the 2-D code
    Z = torch.tensor(Z, dtype=torch.float32)
    g = np.random.RandomState(0).permutation(len(y))
    tr, te = g[:5000], g[5000:]
    d = torch.cdist(Z[te], Z[tr])                              # distances test->train
    nn_idx = d.topk(k, largest=False).indices                 # k nearest train points
    votes = y[tr][nn_idx]
    pred = torch.mode(votes, dim=1).values
    return (pred == y[te]).float().mean().item()

acc_ae  = knn_acc(Z_ae,  y)
acc_pca = knn_acc(Z_pca, y)
print(f"\n2-D code class separation (15-NN accuracy):")
print(f"  autoencoder (nonlinear): {acc_ae:.3f}")
print(f"  PCA (linear, 2 comp):    {acc_pca:.3f}")

# --- 5. Ablation 2: linear autoencoder (remove nonlinearity) -> should match PCA. ---
print("\n=== ablation: LINEAR autoencoder (no nonlinearity, = PCA) ===")
ae_lin = train_ae(linear=True)
with torch.no_grad(): Z_lin = ae_lin.encode(X).cpu().numpy()
print(f"  linear-AE 2-D code 15-NN accuracy: {knn_acc(Z_lin, y):.3f} (~ PCA level)")
# The nonlinear autoencoder's 2-D code separates digits better than PCA's first 2 components
# (Fig. 3); removing the nonlinearity collapses it back to ~PCA.
# (Exact numbers vary by hardware/seed; this is our small run, not the paper's reported result.)

## Visualize the data & results

_Does a deep autoencoder's 2-D code separate MNIST digit classes better than PCA's first two components — and does removing the nonlinearity collapse it back to PCA?_

In [ ]:
import torch, torch.nn as nn, numpy as np
import torchvision, torchvision.transforms as T
torch.manual_seed(0); np.random.seed(0)

# Deep autoencoder 2-D code vs PCA 2-D embedding on an MNIST subset (toy reproduction of Fig. 3),
# plus the linear-autoencoder ablation (= PCA).
class AutoEncoder(nn.Module):
    def __init__(s, code=2, linear=False):
        super().__init__(); act = nn.Identity if linear else nn.ReLU
        s.enc = nn.Sequential(nn.Linear(784,256), act(), nn.Linear(256,64), act(), nn.Linear(64,code))
        s.dec = nn.Sequential(nn.Linear(code,64), act(), nn.Linear(64,256), act(),
                              nn.Linear(256,784), nn.Sigmoid())
    def encode(s, x): return s.enc(x)
    def forward(s, x): return s.dec(s.enc(x))

tf  = T.Compose([T.ToTensor(), T.Lambda(lambda t: t.view(-1))])
raw = torchvision.datasets.MNIST("./data", train=True, download=True, transform=tf)
idx = np.random.RandomState(0).permutation(len(raw))[:6000]
X = torch.stack([raw[i][0] for i in idx]); y = torch.tensor([raw[i][1] for i in idx])

def train_ae(linear=False, epochs=40):
    torch.manual_seed(0); ae = AutoEncoder(linear=linear)
    opt = torch.optim.Adam(ae.parameters(), lr=1e-3); loss_fn = nn.MSELoss(); ae.train(); B=256
    for ep in range(epochs):
        perm = torch.randperm(len(X))
        for s0 in range(0, len(X), B):
            xb = X[perm[s0:s0+B]]; loss = loss_fn(ae(xb), xb)
            opt.zero_grad(); loss.backward(); opt.step()
    return ae

def knn_acc(Z, y, k=15):
    Z = torch.tensor(np.asarray(Z), dtype=torch.float32)
    g = np.random.RandomState(0).permutation(len(y)); tr, te = g[:5000], g[5000:]
    d = torch.cdist(Z[te], Z[tr]); nn_idx = d.topk(k, largest=False).indices
    pred = torch.mode(y[tr][nn_idx], dim=1).values
    return round((pred == y[te]).float().mean().item(), 3)

ae = train_ae(linear=False)
with torch.no_grad(): Z_ae = ae.encode(X).numpy()
Xc = X - X.mean(0); U,S,V = torch.pca_lowrank(Xc, q=2); Z_pca = (Xc @ V).numpy()
ae_lin = train_ae(linear=True)
with torch.no_grad(): Z_lin = ae_lin.encode(X).numpy()

print("autoencoder (nonlinear) 2-D code 15-NN acc:", knn_acc(Z_ae,  y))
print("linear autoencoder (ablation, = PCA)      :", knn_acc(Z_lin, y))
print("PCA (first 2 components)                  :", knn_acc(Z_pca, y))
# Nonlinear autoencoder 2-D code separates digits better than PCA; linear AE collapses to ~PCA.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The headline (codes beat PCA). You squeezed MNIST digits to a 2-D code two ways — the first
            two principal components, and a trained autoencoder's bottleneck — then trained a simple classifier
            on each 2-D code. The autoencoder code gives higher accuracy. What does that demonstrate, and which
            property of the autoencoder is responsible?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compare classifier accuracy on the 2-D PCA code vs the 2-D autoencoder code. — _Both feed a classifier the same number of inputs (2); only the embedding differs, so the gap measures embedding quality._
- Note PCA's code is a linear projection (top-2 variance directions); the autoencoder's is a nonlinear map. — _Digit classes lie on a curved manifold; a flat 2-D plane (PCA) cannot separate them as well as a curved map can._
- Conclude the nonlinearity (the bottleneck trained through nonlinear layers) is what buys the better separation. — _Strip the nonlinearity and the autoencoder collapses back to PCA — same plane, same separation._

**Answer:** It demonstrates the paper's central claim, Fig. 3: a deep nonlinear autoencoder's
                 low-dimensional code separates the classes better than PCA's at the same code size. The
                 responsible property is the nonlinearity in the encoder — it lets the 2-D code follow the
                 curved manifold the digits live on, where PCA is stuck with a flat plane. Our CODEVIZ panel shows
                 the autoencoder 2-D code scatter with visibly tighter digit clusters and a higher classifier
                 accuracy than the 2-D PCA scatter.

</details>

**Problem 2.** Your worked example gave reconstruction $\text{MSE} = 0.015$ for input $x=[1,0,0.8,0.2]$ and output
            $\hat x=[0.9,0.1,0.6,0.2]$. Suppose after more training the decoder improves to
            $\hat x=[1.0,0.0,0.7,0.2]$. What is the new MSE, and did training help?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Per-pixel errors: $x-\hat x = [0,\ 0,\ 0.1,\ 0]$. — _Only the third pixel is still off ($0.8$ vs $0.7$); the rest are now exact._
- Square and sum: $[0,0,0.01,0] \to 0.01$. — _Squared error of the one wrong pixel._
- Divide by 4 pixels: $\text{MSE} = 0.01/4 = 0.0025$. — _Mean over the 4 pixels._

**Answer:** The new $\text{MSE} = 0.0025$, down from $0.015$ — about $6\times$ smaller, so training
                 helped: the reconstruction is much closer to the input. As $\hat x \to x$ the
                 reconstruction loss falls toward $0$; gradient descent on this objective is exactly what drives
                 the decoder (and the code) to improve.

</details>

**Problem 3.** Ablation. You remove every nonlinearity from the autoencoder (so encoder and decoder are pure
            linear layers, code size still 2) and retrain. Its 2-D code now separates the digits no better than
            PCA's first two components — the classifier accuracies are essentially equal. Why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Write the linear encoder/decoder as matrices: $z = W_e x$, $\hat x = W_d z = W_d W_e x$. — _With no activations the whole map is a single rank-2 linear transform $W_d W_e$._
- Recall that the best rank-2 linear reconstruction of the data is the projection onto the top 2 principal directions. — _Minimizing $\sum\lVert x - W_d W_e x\rVert^2$ over rank-2 maps is exactly the problem PCA solves._
- Conclude the linear autoencoder spans the same 2-D subspace as 2-component PCA. — _Same subspace -> same (up to rotation) 2-D embedding -> same class separation._

**Answer:** Because a linear autoencoder with a 2-number code is equivalent to 2-component
                 PCA: the encoder-decoder composes to one rank-2 linear map, and the squared-error-optimal
                 rank-2 linear reconstruction is the projection onto the top two principal directions — exactly
                 what PCA returns. With the nonlinearities gone, the autoencoder can only access flat structure,
                 so it loses its advantage. This is the paper's point in reverse: the nonlinearity is what
                 makes the autoencoder "a nonlinear generalization of PCA," and removing it collapses the two
                 methods together. Our CODEVIZ ablation bar shows the linear-autoencoder accuracy dropping to the
                 PCA level.

</details>